# Environment check
Run these cells top-to-bottom after `docker compose up -d` to confirm HDFS, Spark, and Kafka are all reachable from this notebook.

In [ ]:
# 1. Spark connection (standalone cluster, not local mode)
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("EnvCheck")
    .master("spark://spark-master:7077")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0")
    .getOrCreate()
)
spark

In [ ]:
# 2. HDFS connection
df = spark.createDataFrame([(1, "hello"), (2, "hdfs")], ["id", "msg"])
df.write.mode("overwrite").csv("hdfs://namenode:9000/labs/envcheck")
spark.read.csv("hdfs://namenode:9000/labs/envcheck").show()

In [ ]:
# 3. Kafka connection (produce + consume one message)
!pip install -q kafka-python
from kafka import KafkaProducer, KafkaConsumer
import json, time

producer = KafkaProducer(bootstrap_servers="kafka:9092", value_serializer=lambda v: json.dumps(v).encode())
producer.send("envcheck", {"status": "ok"})
producer.flush()

consumer = KafkaConsumer("envcheck", bootstrap_servers="kafka:9092", auto_offset_reset="earliest", consumer_timeout_ms=5000)
for msg in consumer:
    print(msg.value)
print("Kafka round-trip OK")

In [ ]:
spark.stop()